# 🍎 Shelf Life Prediction of Fruits & Vegetables
## Using ShuffleNet V2 with Transfer Learning

**Capstone Project — VIT-AP University, 2024**  
**Authors:** Satyala Murali Karthik · Mekala Samuel · Yelakanti Ramu  
**Guide:** Dr. S. Kalyani · School of Electronics Engineering

---

### Results
| Metric | Score |
|---|---|
| Accuracy | 97.25% |
| F1 Score | 96.80% |
| ROC AUC | 97.10% |
| Precision-Recall AUC | 95.85% |

**Best Model:** ShuffleNet V2 — lightweight, edge-deployable, outperforms ResNet, MobileNet and EfficientNet

## 1. Install & Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve, accuracy_score, auc
import time
import matplotlib.pyplot as plt
import numpy as np
import os

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

## 2. Hyperparameters

These are entered interactively in the script. Set them here for the notebook run.

In [ ]:
batch_size         = 128
num_epochs         = 20
earlystop_patience = 3
learning_rate      = 0.003
dropout_rate       = 0.5

print(f"Batch size: {batch_size}")
print(f"Epochs: {num_epochs}")
print(f"Learning rate: {learning_rate}")
print(f"Dropout: {dropout_rate}")

## 3. Dataset Loading & Preprocessing

**Dataset structure expected:**
```
data/shelf_life/
    Fresh/     ← fresh produce images
    Rotten/    ← rotten produce images
```

Download from Kaggle: [Fruits and Vegetables Image Recognition Dataset](https://www.kaggle.com/)

**Preprocessing:**
- Resize to 256×256 → CenterCrop to 224×224 (ImageNet standard)
- Normalize with ImageNet mean/std for transfer learning compatibility
- Split: 70% train / 15% validation / 15% test

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Update this path to your dataset location
dataset_path = os.path.join(os.getcwd(), 'data', 'shelf_life')
dataset      = datasets.ImageFolder(root=dataset_path, transform=transform)
num_classes  = len(dataset.classes)

print(f"Classes: {dataset.classes}")
print(f"Total images: {len(dataset)}")

train_size = int(0.70 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f"Train: {train_size} | Val: {val_size} | Test: {test_size}")

## 4. Model — ShuffleNet V2

**Why ShuffleNet V2?**
- Uses channel split + shuffle operations
- ~2-3x fewer FLOPs than ResNet/MobileNet at similar accuracy
- Ideal for edge deployment (Raspberry Pi, mobile)
- Pre-trained on ImageNet — we replace the final FC layer for our classes

In [ ]:
model = models.shufflenet_v2_x1_0(weights='DEFAULT')

# Replace classifier head
model.fc = nn.Sequential(
    nn.Dropout(p=dropout_rate),
    nn.Linear(model.fc.in_features, num_classes)
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

## 5. Training Setup

**Loss:** CrossEntropyLoss — standard for multi-class classification  
**Optimizer:** Adam (lr=0.003) — adaptive learning rate, fast convergence  
**Scheduler:** StepLR (step=7, γ=0.1) — aggressive decay after 7 epochs to refine weights  
**Early Stopping:** patience=3 on validation loss — stops before overfitting

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

class EarlyStopping:
    def __init__(self, patience=7, verbose=False):
        self.patience = patience
        self.verbose  = verbose
        self.counter  = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

def compute_metrics(y_true, y_pred, y_probs):
    acc       = accuracy_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred, average='weighted')
    auc_score = roc_auc_score(y_true, y_probs, multi_class='ovr')
    precision, recall, _ = precision_recall_curve(y_true, y_probs[:, 1], pos_label=1)
    auc_pr = auc(recall, precision)
    return acc, f1, auc_score, precision, recall, auc_pr

print("Training setup complete ✓")

## 6. Training Loop

In [ ]:
early_stopping = EarlyStopping(patience=earlystop_patience, verbose=True)
train_loss_values, val_loss_values = [], []
train_acc_values,  val_acc_values  = [], []
epoch_times = []

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    train_loss, correct_train, total_train = 0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * inputs.size(0)
        _, predicted   = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train   += labels.size(0)

    train_loss    /= total_train
    train_accuracy = correct_train / total_train
    train_loss_values.append(train_loss)
    train_acc_values.append(train_accuracy)

    # Validation
    model.eval()
    val_loss, correct_val, total_val = 0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            val_loss      += loss.item() * inputs.size(0)
            _, predicted   = torch.max(outputs, 1)
            correct_val   += (predicted == labels).sum().item()
            total_val     += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    val_loss    /= total_val
    val_accuracy = correct_val / total_val
    val_loss_values.append(val_loss)
    val_acc_values.append(val_accuracy)

    acc, f1, auc_val, precision, recall, auc_pr = compute_metrics(
        all_labels, all_preds, np.array(all_probs))

    epoch_time = time.time() - start_time
    epoch_times.append(epoch_time)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {epoch_time:.1f}s")
    print(f"  Train  → Loss: {train_loss:.4f}  Acc: {train_accuracy:.4f}")
    print(f"  Val    → Loss: {val_loss:.4f}  Acc: {val_accuracy:.4f}  F1: {f1:.4f}  AUC: {auc_val:.4f}")

    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break
    scheduler.step()

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_loss_values, 'r-', label='Train Loss')
axes[0].plot(val_loss_values,   'b-', label='Val Loss')
axes[0].set_title('Loss vs Epoch'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(train_acc_values, 'r-', label='Train Accuracy')
axes[1].plot(val_acc_values,   'b-', label='Val Accuracy')
axes[1].set_title('Accuracy vs Epoch'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print(f"Average epoch time: {np.mean(epoch_times):.2f}s")

## 8. Save Model

In [ ]:
torch.save(model.state_dict(), 'shufflenet_shelf_life.pth')
print("Model saved → shufflenet_shelf_life.pth")

## 9. Final Test Evaluation

In [ ]:
model.eval()
test_loss, correct_test, total_test = 0, 0, 0
all_test_labels, all_test_preds, all_test_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss    = criterion(outputs, labels)

        test_loss     += loss.item() * inputs.size(0)
        _, predicted   = torch.max(outputs, 1)
        correct_test  += (predicted == labels).sum().item()
        total_test    += labels.size(0)
        all_test_labels.extend(labels.cpu().numpy())
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())

test_loss    /= total_test
test_accuracy = correct_test / total_test

test_acc, test_f1, test_auc, precision, recall, auc_pr = compute_metrics(
    all_test_labels, all_test_preds, np.array(all_test_probs))

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy          : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  F1 Score          : {test_f1:.4f}")
print(f"  ROC AUC           : {test_auc:.4f}")
print(f"  Precision-Recall AUC: {auc_pr:.4f}")